In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_config

In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_helpers

## 1. Read from Bronze

In [0]:
log("Reading from Bronze ...")
df_bronze = spark.table(TBL_BRONZE_RAW)
display(df_bronze.limit(10))


## 2. Extract Customer Columns & Deduplicate

In [0]:
df_customers = df_bronze.select(
    F.col("customer_id"),
    F.col("customer_name"),
    F.col("segment")
    ).distinct()
display(df_customers)

In [0]:
df_duplicate = df_customers.groupBy("customer_id").count().where("count > 1")
df_duplicate.show()

In [0]:
log(f"Rows before duplicate drop: {df_customers.count():,}")
df_silver_customers = df_customers.dropDuplicates(["customer_id"])
log(f"Rows after duplicate drop: {df_silver_customers.count():,}")

## 3. Standardize Segment Values

In [0]:
df_silver_customers = df_silver_customers. \
    withColumn("segment",F.initcap(F.trim(F.col("segment")))
    )

display(df_silver_customers)

## 4. Write to Silver